In [20]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score


In [21]:
import warnings
warnings.filterwarnings("ignore")


In [22]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac1112/doublet_filtered.h5ads/_dataset.h5ads')

In [23]:
df_meta_all = pd.read_csv('/data2st1/junyi/output/atac1112/ATACSC_3REGION_ALL_L2annoated.csv',index_col=0)

In [24]:
df_l3l4 = pd.read_csv('/data2st1/junyi/output/atac1112/iterative/annotated_l3l4.csv',index_col=0)

In [25]:
df_meta_all.head()

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,celltype.L1,celltype.L2,Neurotransmitter_celltype,celltype.L1_ct,Sample_name,Condition,Region,celltype.L2.raw,celltype.L2.refined,region_nt
MC37A_AMY:AAACGAAAGAGTGGAA-1,MC37A_AMY,0.092856,0.130307,2,6,0,0,1,1,5,...,Neuron,AMY Meis1_Abi3bp Glut,Glutamatergic,Glut,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut
MC37A_AMY:AAACGAAAGGGTAGTC-1,MC37A_AMY,0.128610,0.014116,18,7,4,5,5,6,7,...,Immune,Microglia-1,NN,Microglia,MC37A_AMY,MC,AMY,Microglia-1,Perivascular Macrophage,NN
MC37A_AMY:AAACGAAGTACGGAGT-1,MC37A_AMY,0.118241,0.019370,4,1,0,0,1,1,1,...,Neuron,AMY Maf_Pthlh GABA,GABAergic,GABA,MC37A_AMY,MC,AMY,AMY Maf_Pthlh GABA,AMY Maf_Pthlh GABA,AMY_GABA
MC37A_AMY:AAACGAAGTCAGCAAG-1,MC37A_AMY,0.074653,0.063387,6,6,0,0,1,1,5,...,Neuron,AMY Zfhx4_Pde7b GABA,GABAergic,GABA,MC37A_AMY,MC,AMY,AMY Zfhx4_Pde7b GABA,AMY Zfhx4_Pde7b GABA,AMY_GABA
MC37A_AMY:AAACGAAGTCCGAGCT-1,MC37A_AMY,0.098286,0.032532,1,8,0,0,1,1,1,...,Neuron,AMY Meis1_Abi3bp Glut,Glutamatergic,Glut,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut


In [26]:
df_l3l4.head()

,celltype.L3,celltype.L4,celltype.L4.raw
MC37A_AMY:AAAGGATAGCATTGGG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAAACCGTCAGAGTG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAATCGCAGAAAGAG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-0
MC37A_AMY:ACAGAAAGTAGAATAC-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAGCGCAGTGAGCAC-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1


In [27]:
df_meta_all = df_meta_all.merge(df_l3l4[['celltype.L3','celltype.L4']],left_index=True,right_index=True,how='left')

In [28]:
df_meta_all['celltype.L3'] = df_meta_all['celltype.L3'].fillna(df_meta_all['celltype.L2']+"-0").str.replace("/","-").str.replace(" ","_")
df_meta_all['celltype.L4'] = df_meta_all['celltype.L4'].fillna(df_meta_all['celltype.L3']+"-0").str.replace("/","-").str.replace(" ","_")
df_meta_all['celltype.L2'] = df_meta_all['celltype.L2'].str.replace("/","-").str.replace(" ","_")
df_meta_all['Neurotransmitter_celltype'] = df_meta_all['Neurotransmitter_celltype'].fillna("NN")
df_meta_all['region_nt'] = df_meta_all['region_nt'].fillna("NN")


In [29]:
df_meta_all.groupby(['celltype.L3']).size().reset_index(name='count').to_csv("/data2st1/junyi/output/atac1112/iterative/L3_count_merged.csv",index=False)

In [30]:
df_meta_all.groupby(['celltype.L2']).size().reset_index(name='count').to_csv("/data2st1/junyi/output/atac1112/iterative/L2_count_merged.csv",index=False)

In [ ]:
.groupby(['celltype.L4']).size().reset_index(name='count').to_csv("/data2st1/junyi/output/atac1112/iterative/L4_count_merged.csv",index=False)

In [56]:
df_meta_all

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,Neurotransmitter_celltype,celltype.L1_ct,Sample_name,Condition,Region,celltype.L2.raw,celltype.L2.refined,region_nt,celltype.L3,celltype.L4
MC37A_AMY:AAACGAAAGAGTGGAA-1,MC37A_AMY,0.092856,0.130307,2,6,0,0,1,1,5,...,Glutamatergic,Glut,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-1,AMY_Meis1_Abi3bp_Glut-1-0
MC37A_AMY:AAACGAAAGGGTAGTC-1,MC37A_AMY,0.128610,0.014116,18,7,4,5,5,6,7,...,NN,Microglia,MC37A_AMY,MC,AMY,Microglia-1,Perivascular Macrophage,NN,Microglia-1-1,Microglia-1-1-1
MC37A_AMY:AAACGAAGTACGGAGT-1,MC37A_AMY,0.118241,0.019370,4,1,0,0,1,1,1,...,GABAergic,GABA,MC37A_AMY,MC,AMY,AMY Maf_Pthlh GABA,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0
MC37A_AMY:AAACGAAGTCAGCAAG-1,MC37A_AMY,0.074653,0.063387,6,6,0,0,1,1,5,...,GABAergic,GABA,MC37A_AMY,MC,AMY,AMY Zfhx4_Pde7b GABA,AMY Zfhx4_Pde7b GABA,AMY_GABA,AMY_Zfhx4_Pde7b_GABA-1,AMY_Zfhx4_Pde7b_GABA-1-1
MC37A_AMY:AAACGAAGTCCGAGCT-1,MC37A_AMY,0.098286,0.032532,1,8,0,0,1,1,1,...,Glutamatergic,Glut,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-0,AMY_Meis1_Abi3bp_Glut-0-0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MW65A_AMY:TTTGTGTGTTCCTGTC-1,MW65A_AMY,0.145827,0.008648,2,6,0,0,1,1,5,...,GABAergic,GABA,MW65A_AMY,MW,AMY,AMY Rai14_Foxp2 GABA,AMY Rai14_Foxp2 GABA,AMY_GABA,AMY_Rai14_Foxp2_GABA-0,AMY_Rai14_Foxp2_GABA-0-0
MW65A_AMY:TTTGTGTTCAACGTGT-1,MW65A_AMY,0.118720,0.026467,6,6,0,0,1,1,5,...,GABAergic,GABA,MW65A_AMY,MW,AMY,AMY Foxp2_Penk GABA,AMY Foxp2_Penk GABA,AMY_GABA,AMY_Foxp2_Penk_GABA-0,AMY_Foxp2_Penk_GABA-0-1
MW65A_AMY:TTTGTGTTCATACTTC-1,MW65A_AMY,0.136702,0.013699,4,1,0,0,1,1,1,...,GABAergic,GABA,MW65A_AMY,MW,AMY,AMY Maf_Pthlh GABA,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0
MW65A_AMY:TTTGTGTTCTGAGTAC-1,MW65A_AMY,0.128606,0.018868,10,10,5,6,6,7,8,...,NN,Astrocyte,MW65A_AMY,MW,AMY,Astrocyte-1,Astrocyte-1,NN,Astrocyte-1-1,Astrocyte-1-1-1


In [32]:
assert (adata_concat.obs_names == df_meta_all.index).all()
for col in df_meta_all.columns:
    adata_concat.obs[col] = df_meta_all[col]

In [57]:
%time snap.tl.macs3(adata_concat, groupby='celltype.L2',n_jobs=48)

2026-02-11 14:02:08 - INFO - Exporting fragments...
2026-02-11 14:26:47 - INFO - Calling peaks...
100%|██████████| 76/76 [1:15:52<00:00, 59.90s/it] 


CPU times: user 1h 13min 22s, sys: 1h 3min 6s, total: 2h 16min 28s
Wall time: 1h 41min 3s


In [58]:
%time peaks = snap.tl.merge_peaks(adata_concat.uns['macs3'], snap.genome.mm10)


CPU times: user 1min 2s, sys: 5.28 s, total: 1min 7s
Wall time: 20.1 s


In [59]:
peaks.shape

(1071889, 77)

In [ ]:
snap.metrics.summary_by_chrom(adata_concat)

AnnDataSet object with n_obs x n_vars = 176318 x 526765 backed at '/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads'
contains 18 AnnData objects with keys: 'MC37A_AMY', 'MC39C_HIP', 'MC48D_HIP', 'MC50B_AMY', 'MC50B_HIP', 'MC50B_PFC', 'MC52E_AMY', 'MC52E_PFC', 'MW45A_HIP', 'MW47A_AMY', 'MW47A_HIP', 'MW47A_PFC', 'MW51A_AMY', 'MW51A_HIP', 'MW51A_PFC', 'MC25A_PFC', 'MW26A_PFC', 'MW65A_AMY'
    obs: 'sample', 'doublet_probability', 'doublet_score', 'leiden', 'leiden_default', 'leiden_res_0.1', 'leiden_res_0.2', 'leiden_res_0.3', 'leiden_res_0.4', 'leiden_res_0.5', 'leiden_res_0.6', 'leiden_res_0.7', 'leiden_res_0.8', 'leiden_res_0.9', 'leiden_res_1.0', 'leiden_res_1.1', 'leiden_res_1.2', 'leiden_res_1.3', 'leiden_res_1.4', 'leiden_res_1.5', 'leiden_res_1.6', 'leiden_res_1.7', 'leiden_res_1.8', 'leiden_res_1.9', 'celltype.L2.Condition', 'celltype.L1', 'celltype.L2', 'Neurotransmitter_celltype', 'celltype.L1_ct', 'Sample_name', 'Condition', 'Region', 'celltype.L2.raw', 'r

KeyboardInterrupt: 

In [61]:
peaks.to_pandas().to_csv('/data2st1/junyi/output/atac1112/iterative/peaks_L2.csv')

In [62]:
black_list = ['Immune','OPC-Oligo','Doublet','PFC Doublet','PFC Not sure','Not sure','Astro-Epen']

In [ ]:
from snapatac2._snapatac2 import read_motifs, PyDNAMotif

def cis_bp_mouse(unique: bool = True , path="data/motifdb/Mus_musculus.meme") -> list[PyDNAMotif]:
    motifs = read_motifs(path)
    for motif in motifs:
        motif.name = motif.id.split('+')[0]
    if unique:
        unique_motifs = {}
        for motif in motifs:
            name = motif.name
            if (
                    name not in unique_motifs or 
                    unique_motifs[name].info_content() < motif.info_content()
               ):
               unique_motifs[name] = motif
        motifs = list(unique_motifs.values())
    return motifs


In [63]:
%time peak_mat = snap.pp.make_peak_matrix(adata_concat, use_rep=peaks['Peaks'])


CPU times: user 2h 14min 17s, sys: 42min 8s, total: 2h 56min 26s
Wall time: 7min 32s


In [64]:
adata_concat.obsm

AxisArrays (row) with keys: X_spectral, X_umap

In [65]:
peak_mat.obsm['X_umap'] = adata_concat.obsm['X_umap']
peak_mat.obsm['X_spectral'] = adata_concat.obsm['X_spectral']


In [66]:
peak_mat.layers['count'] = peak_mat.X.copy()
sc.pp.normalize_total(peak_mat)
sc.pp.log1p(peak_mat)
peak_mat.obs['expriment']= peak_mat.obs['sample'].str[:2]
peak_mat.write(f"output/atac1112/3REGIONS_peak_l2.h5ads")


... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical
... storing 'expriment' as categorical


In [93]:
adata_concat.close()

In [68]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads')

In [94]:
adata_concat

Closed AnnDataSet object

In [ ]:
%time dmr_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data1st2/hannan_25/data/Nanopore_processV1/nanopore_08_differential/summary/dmr_seg_anno_2tools_nofilter.bed')


In [ ]:
dmr_mat.write(f"output/atac0627/3REGIONS_dmr_new.h5ads")

In [ ]:
adata_concat.close()

In [ ]:
adata_concat

In [ ]:
dmr_mat = sc.read_h5ad(f"output/atac0627/3REGIONS_dmr.h5ads",backed='r')

In [ ]:
dmr_mat

In [ ]:
peak_mat.obs.columns

In [ ]:
peak_mat.obs['celltype.L1nt'] = peak_mat.obs['celltype.L1'].astype('str')
peak_mat.obs.loc[peak_mat.obs['celltype.L1'] == 'Neuron', 'celltype.L1nt'] = peak_mat.obs.loc[peak_mat.obs['celltype.L1'] == 'Neuron', 'region_nt'].astype('str').values
peak_mat.obs['celltype.L1nt'] = peak_mat.obs['celltype.L1nt'].astype('category')

In [ ]:
peak_mat.write(f"output/atac0627/3REGIONS_peak.h5ads")


In [ ]:
peak_mat.var['chr'] = peak_mat.var.index.str.split(r'[:-]').str[0]
peak_mat.var['start'] = peak_mat.var.index.str.split(r'[:-]').str[1]
peak_mat.var['end'] = peak_mat.var.index.str.split(r'[:-]').str[2]

In [ ]:
peak_mat.var.to_csv('/data2st1/junyi/output/atac0627/cCRE/peak.bed', index=False, sep="\t", header=False)

In [ ]:
#celltypes = adata.obs["celltype.L1.tab"].unique()
regions = ['AMY','HIP','PFC']
celltypes = ['OPC-Oligo', 'Immune','Astro-Epen','Neuron']
print(celltypes)
print(regions)

In [ ]:
# # Column to use for stratification
# stratify_column = 'sample'

# # Number of cells to sample from each group
# n_samples_per_group = 2000

# # Perform stratified sampling
# sampled_indices = (
#     adata.obs
#     .groupby(stratify_column, group_keys=False)
#     .apply(lambda x: x.sample(min(n_samples_per_group, len(x))))
#     .index
# )


In [ ]:
# region = 'ALL'
# celltype = 'ALL'
# base_name = f"{region}_{celltype}"
# adata_AMY_neuron  = adata[sampled_indices, :]

# adata_AMY_neuron.obs['expriment'] = adata_AMY_neuron.obs['sample'].str[:2]